# Scopus ASJC citation analysis — pilot\nScopus citing export must already be filtered to **Source type: Journal**. Target publications 2022–2025; citing years unrestricted. Journal self-citations excluded. ASJC frequencies are non-exclusive.\n

In [ ]:
from pathlib import Path
import re, json, hashlib, unicodedata
from collections import Counter, defaultdict
import pandas as pd

BASE = Path.cwd()
if not (BASE / "reference").exists() and (BASE.parent / "reference").exists():
    BASE = BASE.parent
REF, INPUT, OUTPUT = (BASE / d for d in ("reference", "input", "output"))
OUTPUT.mkdir(exist_ok=True)
sources = pd.read_excel(REF / "Scopus_Sources_ASJC_Sep_2026.xlsx", dtype=str).fillna("")
codes = pd.read_excel(REF / "ASJC Classification Codes.xlsx", dtype=str).fillna("")
targets = pd.read_excel(REF / "target_journals.xlsx", dtype=str).fillna("")
citing = pd.read_csv(INPUT / "ukrainian_geographical_journal.csv", dtype=str, low_memory=False).fillna("")
print("Sources", len(sources), "ASJC entries", len(codes), "Targets", len(targets), "Citing records", len(citing))
print("Source columns:", sources.columns.tolist())
print("Target columns:", targets.columns.tolist())
print("Citing columns:", citing.columns.tolist())

In [ ]:
def norm_title(x):
    return re.sub(r"[^\w]+", " ", unicodedata.normalize("NFKC", str(x)).casefold()).strip()
def norm_issn(x):
    s = re.sub(r"[^0-9Xx]", "", str(x)).upper()
    return s if len(s) == 8 else ""
def split_codes(x):
    return sorted(set(re.findall(r"\b\d{4}\b", str(x))))
def col(df, options):
    lookup = {norm_title(c): c for c in df.columns}
    for name in options:
        if norm_title(name) in lookup:
            return lookup[norm_title(name)]
    raise KeyError(f"Missing one of {options}; columns: {list(df.columns)}")

sid = col(sources, ["Sourcerecord ID", "Source ID"])
stitle = col(sources, ["Source Title"])
sissn = col(sources, ["ISSN"])
seissn = col(sources, ["EISSN", "E-ISSN"])
sasc = col(sources, ["All Science Journal Classification Codes (ASJC)"])
ttitle = col(targets, ["journal_title", "Titles", "Title", "Journal", "Source Title"])
tid = col(targets, ["source_id", "Scopus ID", "Source ID", "Sourcerecord ID"])
target = targets.loc[targets[ttitle].map(norm_title) == norm_title("Ukrainian Geographical Journal")]
assert len(target) == 1, "Target journal not uniquely found"
target_id = re.search(r"\d{8,}", str(target.iloc[0][tid]))
assert target_id, "Cannot parse target Source ID"
target_id = target_id.group(0)
target_source = sources.loc[sources[sid].astype(str).str.strip() == target_id]
assert len(target_source) == 1, "Target not uniquely found in source list"
target_codes = set(split_codes(target_source.iloc[0][sasc]))
assert target_codes, "Target ASJC codes missing"

by_issn, by_title = defaultdict(set), defaultdict(set)
source_info = {}
for _, s in sources.iterrows():
    ident = str(s[sid]).strip()
    if not ident: continue
    source_info[ident] = {"title": s[stitle], "codes": split_codes(s[sasc])}
    for field in (sissn, seissn):
        v = norm_issn(s[field])
        if v: by_issn[v].add(ident)
    t = norm_title(s[stitle])
    if t: by_title[t].add(ident)

rows = []
for i, r in citing.iterrows():
    issn = norm_issn(r.get("ISSN", ""))
    title = norm_title(r.get("Source title", ""))
    issn_candidates = by_issn.get(issn, set()) if issn else set()
    title_candidates = by_title.get(title, set()) if title else set()
    candidates = issn_candidates or title_candidates
    method = "issn" if issn_candidates else ("title_exact" if title_candidates else "none")
    if len(candidates) > 1 and title_candidates:
        intersection = candidates & title_candidates
        if intersection: candidates = intersection
    if len(candidates) == 1:
        matched_id = next(iter(candidates))
        status = "self_citation" if matched_id == target_id else "matched"
        asjc = source_info[matched_id]["codes"]
    else:
        matched_id = ""
        status = "ambiguous" if candidates else "unmatched"
        asjc = []
    rows.append({"row_number": i+2, "eid": r.get("EID", ""), "citing_year": r.get("Year", ""),
                 "source_title": r.get("Source title", ""), "issn": r.get("ISSN", ""),
                 "match_status": status, "match_method": method, "matched_source_id": matched_id,
                 "candidate_source_ids": "|".join(sorted(candidates)),
                 "asjc_codes": "|".join(asjc)})
log = pd.DataFrame(rows)
print(log.match_status.value_counts(dropna=False))
assert len(log) == len(citing)

In [ ]:
eligible = log.loc[log.match_status == "matched"].copy()
eligible["codes"] = eligible.asjc_codes.map(lambda x: set(x.split("|")) - {""})
with_codes = eligible.loc[eligible.codes.map(bool)].copy()
denom = len(with_codes)
counter = Counter(c for cs in with_codes.codes for c in cs)
freq = pd.DataFrame([{"asjc_code": c, "citing_documents": n,
                      "percent_of_matched_with_asjc": round(100*n/denom, 2) if denom else 0}
                     for c,n in counter.most_common()])
comparison = pd.DataFrame([{"target_asjc_code": c, "citing_documents": counter.get(c, 0),
                            "percent_of_matched_with_asjc": round(100*counter.get(c,0)/denom,2) if denom else 0}
                           for c in sorted(target_codes)])
overlap = sum(bool(cs & target_codes) for cs in with_codes.codes)
summary = pd.DataFrame([{"target": "Ukrainian Geographical Journal",
                         "source_id": target_id, "citing_records": len(log),
                         "self_citations": int((log.match_status=="self_citation").sum()),
                         "matched": len(eligible), "matched_with_asjc": denom,
                         "unmatched": int((log.match_status=="unmatched").sum()),
                         "ambiguous": int((log.match_status=="ambiguous").sum()),
                         "overlap_records": overlap,
                         "overlap_percent": round(100*overlap/denom,2) if denom else 0,
                         "source_type_filter": "Journal (applied in Scopus)",
                         "citing_year_restriction": "none"}])
review = log.loc[log.match_status.isin(["unmatched", "ambiguous"])].copy()
for name, df in {"pilot_summary.csv": summary, "pilot_asjc_frequencies.csv": freq,
                 "pilot_target_category_comparison.csv": comparison,
                 "pilot_matching_log.csv": log.drop(columns=["codes"], errors="ignore"),
                 "pilot_review_sources.csv": review}.items():
    df.to_csv(OUTPUT/name, index=False)
files = [REF/"Scopus_Sources_ASJC_Sep_2026.xlsx", REF/"ASJC Classification Codes.xlsx",
         REF/"target_journals.xlsx", INPUT/"ukrainian_geographical_journal.csv"]
metadata = {"source_list_snapshot": "September 2026", "source_type_filter": "Journal (Scopus)",
            "target_publication_years": "2022-2025", "citing_years": "unrestricted",
            "input_sha256": {str(p.relative_to(BASE)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}}
(OUTPUT/"pilot_run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
display(summary)
display(comparison)